In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/FYP

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FYP


In [3]:
import pandas as pd

# Load the CSV files into pandas DataFrames
data = pd.read_csv('trawlers.csv')

In [4]:
print(data.shape[0])

4369101


In [5]:
data_sorted=pd.read_csv('mydata.csv')

In [6]:
import pandas as pd
from geopy.distance import geodesic

# Assuming 'data' is the concatenated DataFrame with the gear_type column added
# Sort the data by MMSI and timestamp
data = data.sort_values(by=['mmsi', 'timestamp'])

# Group the data by MMSI
grouped = data.groupby('mmsi')

In [7]:
data_sorted['distance_from_shore']=data['distance_from_shore']
data_sorted['distance_from_port']=data['distance_from_port']
data_sorted['course']=data['course']

In [ ]:
print(data_sorted.head())

           mmsi     timestamp  speed        lat       lon  is_fishing  \
0  1.252340e+12  1.325376e+09    0.0  52.458649  4.581200        -1.0   
1  1.252340e+12  1.325378e+09    0.0  52.458668  4.581167        -1.0   
2  1.252340e+12  1.325379e+09    0.0  52.458633  4.581183        -1.0   
3  1.252340e+12  1.325380e+09    0.0  52.458649  4.581234        -1.0   
4  1.252340e+12  1.325381e+09    0.0  52.458649  4.581183        -1.0   

   distances    headings  distance_from_shore  distance_from_port  course  
0   3.106843  313.161342                  0.0                 0.0   153.0  
1   3.985227  163.500463                  0.0                 0.0   153.0  
2   3.803353   63.427922                  0.0                 0.0   153.0  
3   3.403312  270.000020                  0.0                 0.0   153.0  
4   5.966527   22.299895                  0.0                 0.0   153.0  


In [ ]:
#NTM
import pandas as pd
from geopy.distance import geodesic

# Sort the data by 'mmsi' and 'timestamp'
data_sorted = data.sort_values(by=['mmsi', 'timestamp'])

# Calculate distances between consecutive points
distances = []
for mmsi, group in data_sorted.groupby('mmsi'):
    latitudes = group['lat'].tolist()
    longitudes = group['lon'].tolist()
    dist_list = [geodesic((latitudes[i], longitudes[i]), (latitudes[i + 1], longitudes[i + 1])).meters for i in range(len(latitudes) - 1)]
    dist_list.append(0)  # Add 0 for the last row of each MMSI
    distances.extend(dist_list)

# Add 'distances' column to the DataFrame
data_sorted['distances'] = distances

# Print the first few rows of the updated DataFrame
print(data_sorted.head())

           mmsi     timestamp  distance_from_shore  distance_from_port  speed  \
0  9.924005e+12  1.379601e+09                  0.0         1414.178833    0.0   
1  9.924005e+12  1.379602e+09                  0.0         1414.178833    0.0   
2  9.924005e+12  1.379604e+09                  0.0         1414.178833    0.1   
3  9.924005e+12  1.379605e+09                  0.0         1414.178833    0.1   
4  9.924005e+12  1.379608e+09                  0.0         1414.178833    0.0   

       course       lat        lon  is_fishing           source  distances  
0  298.500000  8.861500 -79.668427        -1.0  false_positives   1.833676  
1  298.500000  8.861506 -79.668442        -1.0  false_positives   5.062912  
2  128.399994  8.861511 -79.668488        -1.0  false_positives   0.839230  
3  111.199997  8.861511 -79.668480        -1.0  false_positives   2.729710  
4   41.700001  8.861502 -79.668503        -1.0  false_positives   2.623719  


In [ ]:
#NTM
data_sorted = data_sorted.drop(columns=['source'])

In [ ]:
# Check number of NaN values before applying AR
nan_count_before = data_sorted.isnull().sum()

# Your AR model code here

print(f"Number of NaN values before AR:\n{nan_count_before}")

Number of NaN values before AR:
mmsi                    0
timestamp               0
speed                  78
lat                     0
lon                     0
is_fishing              0
distances               0
headings                1
distance_from_shore     0
distance_from_port      0
course                 78
dtype: int64


In [ ]:
data1=data_sorted

In [8]:
clean_data = data_sorted.dropna()

In [9]:
from statsmodels.tsa.ar_model import AutoReg
import pandas as pd

# Assume data_sorted is your DataFrame containing the ship data

# Columns to exclude (non-numeric or target columns)
exclude_cols = ['mmsi', 'timestamp']

# Iterate over each column (excluding non-numeric and target columns)
for col in data_sorted.columns:
    if col not in exclude_cols:
        # Train AR model
        model = AutoReg(data_sorted[col].dropna(), lags=1)  # Assuming lag of 1 for simplicity
        model_fit = model.fit()

        # Predict missing values in the original dataset
        nan_indices = data_sorted[col].isnull()
        nan_count = nan_indices.sum()
        if nan_count > 0:
            forecast = model_fit.predict(start=len(data_sorted) + nan_count - 1, end=len(data_sorted) + 2*nan_count - 2)
            data_sorted.loc[nan_indices, col] = forecast[-nan_count:]
            print(f"Filled {nan_count} missing values in column {col} using AR model")


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/deterministic.py:307: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_inde

Filled 78 missing values in column speed using AR model
Filled 1 missing values in column headings using AR model


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Filled 78 missing values in column course using AR model


/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/deterministic.py:307: UserWarning: Only PeriodIndexes, DatetimeIndexes with a frequency set, RangesIndexes, and Index with a unit increment support extending. The index is set will contain the position relative to the data length.
  fcast_index = self._extend_index(index, steps, forecast_index)


In [ ]:

# Check number of NaN values after applying AR
nan_count_after = clean_data.isnull().sum()

print(f"Number of NaN values after AR:\n{nan_count_after}")

Number of NaN values after AR:
mmsi                   0
timestamp              0
speed                  0
lat                    0
lon                    0
is_fishing             0
distances              0
headings               0
distance_from_shore    0
distance_from_port     0
course                 0
dtype: int64


In [ ]:
#NTM
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')  # Replace missing values with median
data_sorted = imputer.fit_transform(data_sorted)

In [ ]:
# NTM Check the unique values in the 'source' feature
unique_sources = data['source'].unique()
print(unique_sources)

from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'source' feature to encode the values
data_sorted['source'] = label_encoder.fit_transform(data_sorted['source'])

['false_positives' 'dalhousie_ps' 'gfw' 'crowd_sourced']


In [10]:
# Pre processing function
def preprocess_data(data_sorted):
    label_encoder = LabelEncoder()
    features = []
    labels = []
    for index, row in data_sorted.iterrows():
        features.append({
            'mmsi': row['mmsi'],
            'timestamp':row['timestamp'],
            'lat': row['lat'],
            'lon': row['lon'],
            'course': row['course'],
            'distance_from_shore': row['distance_from_shore'],
            'distance_from_port': row['distance_from_port'],
            'speed': row['speed'],
            'distances': row['distances'],
            'headings': row['headings'],
        })
        labels.append(1 if row['is_fishing'] > 0 else 0)

    features_df = pd.DataFrame(features)

    return features_df, np.array(labels)

In [ ]:
#NTM
import pandas as pd
def check_data_types(data):
  for col in data.columns:
    print(f"Feature: {col}, Type: {data[col].dtype}")
# Convert numerical features to float64
numerical_features = [col for col in data_sorted.columns]

data_sorted = data_sorted.astype({col: 'float64' for col in numerical_features})

print("Data types after conversion:")
check_data_types(data_sorted.copy())

Data types after conversion:
Feature: mmsi, Type: float64
Feature: timestamp, Type: float64
Feature: distance_from_shore, Type: float64
Feature: distance_from_port, Type: float64
Feature: speed, Type: float64
Feature: course, Type: float64
Feature: lat, Type: float64
Feature: lon, Type: float64
Feature: is_fishing, Type: float64
Feature: source, Type: float64
Feature: distances, Type: float64


In [35]:
from sklearn.preprocessing import StandardScaler
def preprocess_data(data_sorted):
    features = data_sorted[['mmsi', 'lat', 'lon', 'course', 'speed', 'distances', 'distance_from_shore', 'distance_from_port', 'headings', 'course']].values
    labels = data_sorted['is_fishing'].values

    # Normalize numerical features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Reshape features for LSTM input (samples, timesteps, features)
    features_reshaped = []
    max_timesteps = 0
    for mmsi in data_sorted['mmsi'].unique():
        ship_data = features_scaled[data_sorted['mmsi'] == mmsi]
        features_reshaped.append(ship_data)
        max_timesteps = max(max_timesteps, ship_data.shape[0])

    # Pad sequences to have the same length
    for i in range(len(features_reshaped)):
        features_reshaped[i] = np.pad(features_reshaped[i], ((0, max_timesteps - features_reshaped[i].shape[0]), (0, 0)), mode='constant', constant_values=0)

    features_reshaped = np.array(features_reshaped)

    # Ensure that the number of samples in features_reshaped matches the number of labels
    labels = labels[:features_reshaped.shape[0]]

    return features_reshaped, labels, scaler



In [36]:
# Preprocess the data
import numpy as np
X_train, y_train, scaler = preprocess_data(data_sorted)

In [37]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
# Model configuration
lstm_units = 5
learning_rate = 0.005

# Create the LSTM model
model = Sequential()
model.add(LSTM(units=lstm_units, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32,  validation_split=0.2)

In [ ]:
model.save('ntm.json')
model.save_weights('ntm.h5')

In [11]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Preprocess data by gear_type
preprocessed_data = []
for gear_type, group_data in data_sorted.groupby('mmsi'):
    features, labels = preprocess_data(group_data)  # Call your existing preprocess_data function
    preprocessed_data.extend([(features_row, labels_row) for features_row, labels_row in zip(features.itertuples(index=False), labels)])

In [12]:
import pandas as pd

# Assuming features and labels are your NumPy arrays
features_df = pd.DataFrame(features)
labels_df = pd.DataFrame(labels)

# Save the DataFrames to CSV files
features_df.to_csv('/content/drive/My Drive/FYP/f_df.csv', index=False)
labels_df.to_csv('/content/drive/My Drive/FYP/l_df.csv', index=False)


In [13]:
# Convert features back to a DataFrame
features_df = pd.DataFrame(features)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into training, validation, and testing sets
X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, random_state=42)

In [15]:
# Combine preprocessed features and labels into DataFrames
features_df = pd.DataFrame([data_point[0] for data_point in preprocessed_data])
labels_df = pd.DataFrame([data_point[1] for data_point in preprocessed_data])

# Train the model (example using an LSTM)
from keras.models import Sequential
from keras.layers import LSTM, Dense

In [16]:
import numpy as np

# Convert DataFrame to NumPy array
X_train_array = X_train.values
X_val_array = X_val.values


# Reshape features for LSTM input
X_train_reshaped = X_train_array.reshape((X_train_array.shape[0], X_train_array.shape[1], 1))
X_val_reshaped = X_val_array.reshape((X_val_array.shape[0], X_val_array.shape[1], 1))

In [17]:
from keras.utils import to_categorical

# One-hot encode labels for multi-class classification
y_train_encoded = to_categorical(y_train)
y_val_encoded = to_categorical(y_val)

In [18]:
# Extract the values from the DataFrame
X_train_values = features_df.values

# Reshape X_train_values to be 3D: (num_samples, 1, num_features)
X_train_reshaped = X_train_values.reshape((X_train_values.shape[0], 1, X_train_values.shape[1]))


In [32]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Handling Missing Values:
imputer = SimpleImputer(strategy='median')  # Replace missing values with median
#features_df = imputer.fit_transform(features_df)

# 2. Checking Label Distribution:
num_fishing = (labels > 0).sum()
num_not_fishing = (labels == 0).sum()
num_unsure = (labels == -1).sum()
print("Number of unique MMSIs with fishing labels:")
print(f"  - Fishing: {num_fishing}")
print(f"  - Not Fishing: {num_not_fishing}")
print(f"  - Unsure: {num_unsure}")

# 3. Label Encoding:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)  # Use LabelEncoder for binary classes

# 4. Feature Normalization:
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(features_df)  # Normalize numerical features

# 5. Reshape for LSTM:
#X_train_reshaped = normalized_features.reshape((X_train_reshaped.shape[0], 1, 9))

Number of unique MMSIs with fishing labels:
  - Fishing: 546
  - Not Fishing: 34580
  - Unsure: 0


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

# Model architecture definition
# Model architecture definition
model = Sequential()
# Conv1D layer with 64 filters and a kernel size of 3
model.add(Conv1D(64, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
# MaxPooling1D layer
model.add(MaxPooling1D(2))
# Flatten layer to flatten the output of the convolutional layer
model.add(Flatten())
model.add(Dense(2, activation='softmax'))

from tensorflow.keras.optimizers import Adam

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=10000,
    decay_rate=0.9
)
optimizer = Adam(learning_rate=0.005)

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
13520/87382 [===>..........................] - ETA: 3:15 - loss: nan - accuracy: 0.9828

In [37]:
# Evaluate the model on the testing data
accuracy = model.evaluate(X_val, y_val)

print("Accuracy:", accuracy)

AttributeError: 'LogisticRegression' object has no attribute 'evaluate'

In [36]:
predictions = model.predict(X_val)


NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [34]:
from sklearn.model_selection import train_test_split

# Assuming your preprocessed features and labels are in `features_df` and `labels_df`

# Split data into training, validation, and testing sets (using a different random state for validation)
X_train, X_test, y_train, y_test = train_test_split(features_df, labels_df, test_size=0.2, random_state=123)  # Different random state

# Reshape X_train and X_test for LSTM input (if applicable)
if 'LSTM' in str(type(model)):  # Check if model is an LSTM
  X_train_reshaped = X_train.values.reshape((X_train.shape[0], X_train.shape[1], 1))
  X_test_reshaped = X_test.values.reshape((X_test.shape[0], X_test.shape[1], 1))
else:
  X_train_reshaped = X_train.values
  X_test_reshaped = X_test.values

# Evaluate the model on the validation set
model.evaluate(X_test, y_test)

# Print the validation accuracy or other relevant metrics (e.g., loss, F1-score)
print("Validation Accuracy:", model.evaluate(X_test_reshaped, y_test)[1])  # Assuming accuracy metric at index 1

# Optionally, make predictions on new unseen data
predictions = model.predict(X_test_reshaped)

# Further analyze predictions based on your use case (e.g., calculate precision, recall)


AttributeError: 'LogisticRegression' object has no attribute 'evaluate'

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# Model configuration
lstm_units = 5
learning_rate = 0.005

# Create the LSTM model
model = Sequential()
model.add(LSTM(units=lstm_units, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32)


In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 7, 64)             256       
                                                                 
 max_pooling1d (MaxPooling1  (None, 3, 64)             0         
 D)                                                              
                                                                 
 flatten (Flatten)           (None, 192)               0         
                                                                 
 dense (Dense)               (None, 2)                 386       
                                                                 
Total params: 642 (2.51 KB)
Trainable params: 642 (2.51 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
model.save('model3_architecture.json')
model.save_weights('model3_weights.h5')